# Lab 4 · Bias, variance, regularization and explainability
**Slides:** Module 07 (bias and variance, learning curve, regularization, explainable AI) and Module 06 (clustering)  ·  **Time:** 60 min

**You will:** see under- and over-fitting · draw learning curves · regularize a neural network · read partial-dependence plots · cluster fields without labels.

In [ ]:
# --- setup: works locally and in Google Colab ---
import os, sys
if "google.colab" in sys.modules and not os.path.exists("aiml"):
    print("Upload the course kit zip to Colab, then unzip it:  !unzip -q aiml_course_kit.zip && %cd aiml_course_kit")
sys.path.insert(0, os.path.abspath(".."))
sys.path.insert(0, os.path.abspath("."))
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": .3, "axes.spines.top": False, "axes.spines.right": False})
DATA = "../data" if os.path.exists("../data") else "data"

from aiml.data import load_asphaltene, load_eor, load_vfm, ASPH_FEATURES, ASPH_TARGET, VFM_FEATURES
from aiml.models import regressors
from sklearn.model_selection import learning_curve, validation_curve, KFold, GroupKFold
from sklearn.tree import DecisionTreeRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LinearRegression
a = load_asphaltene(drop_suspect=True)

## 1 · Under- and over-fitting on one curve
Fit polynomials of growing degree to one precipitation series (n-C7, 308 K).

In [ ]:
s = a[(a.temperature_K == 308) & (a.alkane_carbon_number == 7)]
x, yv = s[["dilution_ratio"]].values, s[ASPH_TARGET].values; xx = np.linspace(x.min(), x.max(), 200)[:, None]
for d in [1, 2, 4]:
    m = make_pipeline(PolynomialFeatures(d), LinearRegression()).fit(x, yv); plt.plot(xx, m.predict(xx), label=f"degree {d}")
plt.scatter(x, yv, c="k", zorder=3, label="data"); plt.ylim(0, yv.max() * 1.6); plt.legend(); plt.xlabel("dilution ratio"); plt.ylabel("wt%"); plt.show()

## 2 · Validation curve: model complexity

In [ ]:
X, y = a[ASPH_FEATURES].astype(float), a[ASPH_TARGET]; cv = GroupKFold(5); g = a.series
depths = list(range(1, 13))
tr, va = validation_curve(DecisionTreeRegressor(random_state=0), X, y, param_name="max_depth", param_range=depths, cv=cv, groups=g, scoring="r2")
plt.plot(depths, tr.mean(1), "o-", label="train"); plt.plot(depths, va.mean(1), "s-", label="validation (unseen series)")
plt.xlabel("max_depth"); plt.ylabel("R²"); plt.legend(); plt.title("bias ← → variance"); plt.show()

## 3 · Learning curve: would more data help?
VFM data, group 5-fold, extra trees vs linear regression.

In [ ]:
v = load_vfm()
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
for axi, name in zip(ax, ["Linear regression", "Extra trees"]):
    n, tr, va = learning_curve(regressors()[name], v[VFM_FEATURES], v.oil_rate_stb_d, cv=GroupKFold(5), groups=v.case_id, train_sizes=np.linspace(.1, 1, 6), scoring="r2")
    axi.plot(n, tr.mean(1), "o-", label="train"); axi.plot(n, va.mean(1), "s-", label="validation"); axi.set_title(name); axi.set_xlabel("training rows"); axi.legend()
plt.tight_layout(); plt.show()

Linear regression: train and validation meet early at a low score → **high bias**, more data will not help, a better model will.
Extra trees: a gap that shrinks with data → **variance**, more tests still help.

### ✏️ Your turn 1 · regularization
Fit `MLPRegressor(hidden_layer_sizes=(64, 64), solver="lbfgs", max_iter=5000)` on the asphaltene data with
`alpha` in `[1e-5, 1e-3, 1e-1, 1, 10]` and draw the group-CV validation curve (parameter `mlpregressor__alpha`).

In [ ]:
nn = make_pipeline(StandardScaler(), MLPRegressor(hidden_layer_sizes=(64, 64), solver="lbfgs", max_iter=5000, random_state=0))
alphas = [1e-5, 1e-3, 1e-1, 1, 10]

## 4 · Explainable AI: partial dependence
How does the model's prediction change with one input, averaged over the others?

In [ ]:
from sklearn.inspection import PartialDependenceDisplay
gb = regressors()["Gradient boosting"].fit(X, y)
PartialDependenceDisplay.from_estimator(gb, X, ASPH_FEATURES, kind="both", subsample=60, random_state=0)
plt.gcf().set_size_inches(14, 4); plt.tight_layout(); plt.show()

Each panel is an engineering check: precipitation should **rise with dilution**, **fall with carbon number** (heavier
alkanes are better solvents) and fall with temperature. A model that is accurate but violates physics is not ready.
The course notebooks use **SHAP** for the same purpose (`pip install shap`): `shap.Explainer(model.predict, X)`.

## 5 · Unsupervised: clustering EOR projects without labels

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
e = load_eor(log_viscosity=True); F = ["log_viscosity", "api_gravity", "depth_ft", "log_permeability"]
Z = StandardScaler().fit_transform(e[F])
ks = range(2, 9); sil = [silhouette_score(Z, KMeans(k, n_init=10, random_state=0).fit_predict(Z)) for k in ks]
plt.plot(ks, sil, "o-"); plt.xlabel("k"); plt.ylabel("silhouette"); plt.show()
e["cluster"] = KMeans(3, n_init=10, random_state=0).fit_predict(Z)
pd.crosstab(e.cluster, e.eor_method)

### ✏️ Your turn 2
Describe each of the three clusters in engineering terms (median API, viscosity, depth). Do the clusters match
"thermal", "gas" and "chemical" EOR?

## Key ideas
* Validation curves show complexity; learning curves show whether data or model is the bottleneck.
* Regularization (alpha, depth, number of neighbours) is the dial between bias and variance.
* Partial dependence and SHAP turn a black box into curves an engineer can check against physics.
* Clustering finds structure without labels; labels then tell you what the clusters mean.

In [ ]:
# e.groupby('cluster')[...].median()